In [ ]:
import json
from tqdm import tqdm
import pandas as pd
from glob import glob

folders = glob('../helm_lite_v1.13.0/runs/*/*')

In [ ]:
dfs = []
for i in tqdm(range(len(folders))):
    with open(f'{folders[i]}/display_predictions.json', 'r') as f:
        data = json.load(f)
    
    temp1 = pd.json_normalize(data, sep='_')
    with open(f'{folders[i]}/display_requests.json', 'r') as f:
        data = json.load(f)

    temp2 = pd.json_normalize(data, sep='_')

    with open(f'{folders[i]}/instances.json', 'r') as f:
        data = json.load(f)

    temp3 = pd.json_normalize(data, sep='_')
    temp3 = temp3.rename(columns={'id': 'instance_id'})
    
    df = (
        temp1
        .merge(temp2, on='instance_id', how='left')
        .merge(temp3, on='instance_id', how='left')
    )
    df['folder'] = folders[i]
    df['task'] = folders[i].split("/")[-1].split(":", 1)[0]
    dfs.append(df)

In [ ]:
combined = pd.concat(dfs, axis=0, ignore_index=True, join='outer', sort=False)

In [ ]:
for model_name, group in combined.groupby("request_model"):
    model_name = model_name.replace('/', '_')
    group.to_pickle(f"../helmBenchmark/{model_name!s}.pkl")